# 03 - Fine-tune on Colab (free GPU)

Runs the SFT pipeline on a Colab T4 instead of your laptop.

**Before running:** set the runtime to GPU via **Runtime -> Change runtime type -> Hardware accelerator: GPU**, then run the cells top to bottom.

## 1. Confirm a GPU is attached

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime -> GPU")

## 2. Clone the repo

In [ ]:
!git clone https://github.com/siddharth0998/LLM-from-scratch.git
%cd LLM-from-scratch

## 3. Install dependencies

Colab already ships torch; we just add the project deps.

In [ ]:
!pip install -q tiktoken transformers pyyaml

## 4. Prepare the data

Start with a subset for a quick first run. Remove `--limit` for the full 52k.

In [ ]:
!python scripts/prepare_sft_data.py --limit 5000

## 5. Smoke test (optional but recommended)

Overfits a handful of examples; the loss should drop toward ~0.

In [ ]:
!python scripts/train_sft.py --smoke

## 6. Fine-tune

Reads `configs/sft.yaml`, trains, and saves the best model to `checkpoints/gpt2-sft.pth`.

In [ ]:
!python scripts/train_sft.py

## 7. Quick chat test with the fine-tuned model

In [ ]:
import sys; sys.path.insert(0, "src")
import torch
from gpt2_chatbot.model import GPTModel, get_config
from gpt2_chatbot.tokenizer import get_tokenizer, text_to_token_ids, token_ids_to_text, get_eot_id
from gpt2_chatbot.inference import generate
from gpt2_chatbot.data import render_for_inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = get_config("gpt2-small (124M)", context_length=1024, qkv_bias=True, drop_rate=0.0)
model = GPTModel(cfg)
model.load_state_dict(torch.load("checkpoints/gpt2-sft.pth", map_location=device))
model.to(device).eval()
tok = get_tokenizer()

prompt = render_for_inference([{"role": "user", "content": "Give three tips for staying healthy."}])
ids = text_to_token_ids(prompt, tok).to(device)
out = generate(model, ids, max_new_tokens=120, context_size=cfg["context_length"],
               top_k=40, temperature=0.8, eos_id=get_eot_id(tok))
print(token_ids_to_text(out, tok)[len(prompt):])

## 8. Download the checkpoint to your machine

In [ ]:
from google.colab import files
files.download("checkpoints/gpt2-sft.pth")